In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================
# Transfer: PCam SSL pretrain (SimCLR) -> Colorectal linear probe (ResNet-18)
# Protocol: 1 split x 3 seeds (n=3), label_fracs 1/5/10
# Outputs:
#   results_raw_resnet_transfer_colorectal.csv
#   results_summary_resnet_transfer_colorectal.csv + .tex
# ============================================================

import os, time, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision
from torchvision import transforms
from PIL import Image

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import label_binarize

OUT_DIR = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)

SPLIT_ID = 0
SEEDS = [0,1,2]
LABEL_FRACS = [0.01, 0.05, 0.10]

# SSL pretrain on PCam
SSL_EPOCHS = 10
SSL_BATCH = 128
SSL_LR = 1e-3
TEMPERATURE = 0.2

# Probe on colorectal (multi-class)
PROBE_EPOCHS = 15
PROBE_BATCH = 256
PROBE_LR = 1e-3

# Speed knobs (optional)
PCAM_SAMPLE_FRAC = 0.2     # increase later if you want
COLO_SAMPLE_FRAC = 1.0     # keep 1.0 usually ok (tiles_5000)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def append_row_csv(path, row: dict):
    df = pd.DataFrame([row])
    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)

def summarize_mean_std(raw_csv_path, group_cols, metric_cols, out_csv_path, out_tex_path=None):
    df = pd.read_csv(raw_csv_path)
    g = df.groupby(group_cols)[metric_cols].agg(["mean","std","count"]).reset_index()
    g.columns = ["_".join([c for c in col if c]) for col in g.columns.values]
    g.to_csv(out_csv_path, index=False)
    if out_tex_path:
        g.to_latex(out_tex_path, index=False, float_format="%.4f")
    return g

# -----------------------
# Find PCam
# -----------------------
def find_pcam_root():
    for d in os.listdir("/kaggle/input"):
        root = os.path.join("/kaggle/input", d)
        if os.path.exists(os.path.join(root, "train_labels.csv")) and os.path.isdir(os.path.join(root, "train")):
            return root
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train_labels.csv" in files and "train" in dirs:
            return root
    raise FileNotFoundError("PCam not found. Add histopathologic-cancer-detection.")

PCAM_ROOT = find_pcam_root()
pcam_df = pd.read_csv(os.path.join(PCAM_ROOT, "train_labels.csv"))
train_dir = os.path.join(PCAM_ROOT, "train")

pcam_df["path"] = pcam_df["id"].apply(lambda x: os.path.join(train_dir, f"{x}.tif"))
if not os.path.exists(pcam_df["path"].iloc[0]):
    pcam_df["path"] = pcam_df["id"].apply(lambda x: os.path.join(train_dir, f"{x}.png"))
pcam_df = pcam_df[pcam_df["path"].apply(os.path.exists)].reset_index(drop=True)

if 0 < PCAM_SAMPLE_FRAC < 1.0:
    pcam_df = pcam_df.sample(frac=PCAM_SAMPLE_FRAC, random_state=42).reset_index(drop=True)

pcam_paths = pcam_df["path"].values
pcam_y = pcam_df["label"].values.astype(int)
print("PCam:", len(pcam_paths))

# -----------------------
# Robust: Find Colorectal tiles_5000 (handles nested folder)
# -----------------------
from pathlib import Path

def find_colorectal_tiles_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        for d in dirs:
            if d.lower() == "kather_texture_2016_image_tiles_5000":
                return os.path.join(root, d)
    raise FileNotFoundError("Colorectal tiles_5000 not found. Add colorectal-histology-mnist dataset.")

COLO_DIR = find_colorectal_tiles_dir()
print("COLO_DIR:", COLO_DIR)

def list_images_recursively(base_dir):
    exts = {".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}
    paths = []
    for p in Path(base_dir).rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            paths.append(str(p))
    return paths

# If no images directly/recursively, try descending one folder (nested structure)
img_paths = list_images_recursively(COLO_DIR)
if len(img_paths) == 0:
    children = [p for p in Path(COLO_DIR).iterdir() if p.is_dir()]
    if len(children) == 1:
        COLO_DIR = str(children[0])
        print("Nested COLO_DIR detected, switching to:", COLO_DIR)
        img_paths = list_images_recursively(COLO_DIR)

if len(img_paths) == 0:
    raise RuntimeError(f"No images found under {COLO_DIR}. Print tree to debug.")

# Infer class = parent folder name of each image
parents = [Path(p).parent.name for p in img_paths]
class_names = sorted(list(set(parents)))
class_to_idx = {c:i for i,c in enumerate(class_names)}

colo_paths = np.array(img_paths)
colo_labels = np.array([class_to_idx[Path(p).parent.name] for p in img_paths], dtype=int)

print("Colorectal tiles:", len(colo_paths), "n_classes:", len(class_names))
print("Example classes:", class_names[:10])


# -----------------------
# Datasets
# -----------------------
ssl_tf = transforms.Compose([
    transforms.RandomResizedCrop(96, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
])

sup_tf_colo = transforms.Compose([
    transforms.Resize((96,96)),
    transforms.ToTensor(),
])

class SSLTwoView(Dataset):
    def __init__(self, paths, tf):
        self.paths = paths
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.tf(img)

class SupMulti(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths = paths
        self.labels = labels
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        x = self.tf(img)
        y = int(self.labels[i])
        return x, y

# -----------------------
# SimCLR model
# -----------------------
class ResNetSimCLR(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        backbone = torchvision.models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])  # [B,512,1,1]
        self.proj = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, out_dim),
        )
    def forward(self, x):
        h = self.encoder(x).flatten(1)
        z = self.proj(h)
        z = F.normalize(z, dim=1)
        return h, z

def nt_xent(z1, z2, T=0.2):
    B = z1.size(0)
    z = torch.cat([z1, z2], 0)  # 2B,D
    sim = (z @ z.t()) / T
    mask = torch.eye(2*B, device=z.device).bool()
    sim = sim.masked_fill(mask, -1e9)
    pos = torch.cat([torch.diag(sim, B), torch.diag(sim, -B)], 0)
    loss = -pos + torch.logsumexp(sim, dim=1)
    return loss.mean()

def pretrain_simclr_on_pcam(seed):
    set_all_seeds(seed)
    ds = SSLTwoView(pcam_paths, ssl_tf)
    dl = DataLoader(ds, batch_size=SSL_BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    model = ResNetSimCLR().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=SSL_LR)

    model.train()
    for ep in range(SSL_EPOCHS):
        total = 0.0; n=0
        for x1,x2 in dl:
            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            _, z1 = model(x1)
            _, z2 = model(x2)
            loss = nt_xent(z1, z2, TEMPERATURE)
            loss.backward()
            opt.step()
            total += float(loss.item()) * x1.size(0); n += x1.size(0)
        print(f"[SSL] seed{seed} ep{ep+1}/{SSL_EPOCHS} loss={total/max(1,n):.4f}")
    return model

# -----------------------
# Probe (multi-class)
# -----------------------
class LinearProbeMC(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_classes)
    def forward(self, h):
        return self.fc(h)

@torch.no_grad()
def extract_feats(encoder, loader):
    encoder.eval()
    X = []
    Y = []
    for x,y in loader:
        x = x.to(device, non_blocking=True)
        h = encoder(x).flatten(1)
        X.append(h.cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(X,0), np.concatenate(Y,0)

def train_probe_mc(encoder, train_loader, val_loader, n_classes, seed):
    Xtr, ytr = extract_feats(encoder, train_loader)
    Xva, yva = extract_feats(encoder, val_loader)

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
    ytr_t = torch.tensor(ytr, dtype=torch.long, device=device)
    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=device)
    yva_t = torch.tensor(yva, dtype=torch.long, device=device)

    probe = LinearProbeMC(Xtr.shape[1], n_classes).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=PROBE_LR)
    ce = nn.CrossEntropyLoss()

    best = 1e9
    best_state = None

    for ep in range(PROBE_EPOCHS):
        probe.train()
        perm = torch.randperm(Xtr_t.size(0), device=device)
        for i in range(0, Xtr_t.size(0), PROBE_BATCH):
            idx = perm[i:i+PROBE_BATCH]
            logits = probe(Xtr_t[idx])
            loss = ce(logits, ytr_t[idx])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        probe.eval()
        with torch.no_grad():
            vloss = float(ce(probe(Xva_t), yva_t).item())
        if vloss < best:
            best = vloss
            best_state = {k: v.detach().cpu() for k,v in probe.state_dict().items()}
        print(f"[Probe] ep{ep+1}/{PROBE_EPOCHS} val_loss={vloss:.4f}")

    probe.load_state_dict(best_state, strict=True)
    return probe

def eval_mc(probe, encoder, test_loader, n_classes):
    Xte, yte = extract_feats(encoder, test_loader)
    Xte_t = torch.tensor(Xte, dtype=torch.float32, device=device)
    with torch.no_grad():
        logits = probe(Xte_t).cpu().numpy()
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)

    preds = probs.argmax(axis=1)
    acc = accuracy_score(yte, preds)
    f1 = f1_score(yte, preds, average="macro")

    # AUROC macro one-vs-rest
    y_bin = label_binarize(yte, classes=list(range(n_classes)))
    try:
        auc = roc_auc_score(y_bin, probs, average="macro", multi_class="ovr")
    except Exception:
        auc = float("nan")

    return {"acc": float(acc), "auc": float(auc), "f1": float(f1)}

# -----------------------
# Split colorectal for transfer evaluation (fixed split across seeds)
# -----------------------
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_rel, test_rel = next(sss.split(np.zeros(len(colo_labels)), colo_labels))
train_paths = colo_paths[train_rel]; train_y = colo_labels[train_rel]
test_paths  = colo_paths[test_rel];  test_y  = colo_labels[test_rel]

# further val split from train
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=123)
tr2_rel, va2_rel = next(sss2.split(np.zeros(len(train_y)), train_y))
tr_paths = train_paths[tr2_rel]; tr_y = train_y[tr2_rel]
va_paths = train_paths[va2_rel]; va_y = train_y[va2_rel]

print("Colo split:", len(tr_paths), len(va_paths), len(test_paths))

# -----------------------
# RUN
# -----------------------
RAW_PATH = f"{OUT_DIR}/results_raw_resnet_transfer_colorectal.csv"

for seed in SEEDS:
    # 1) SSL pretrain on PCam (SimCLR)
    ssl_model = pretrain_simclr_on_pcam(seed)
    encoder = ssl_model.encoder  # use encoder only

    for frac in LABEL_FRACS:
        set_all_seeds(seed + int(frac*1000))

        # stratified subsample of colorectal labeled data
        rng = np.random.RandomState(seed + int(frac*10000))
        # per class sample
        idx = np.arange(len(tr_y))
        # overall stratified via sss train_size
        k = max(1, int(len(tr_y)*frac))
        sss3 = StratifiedShuffleSplit(n_splits=1, train_size=k, random_state=seed + int(frac*1000))
        sub_rel, _ = next(sss3.split(np.zeros(len(tr_y)), tr_y))
        sub_paths = tr_paths[sub_rel]; sub_y = tr_y[sub_rel]

        dl_tr = DataLoader(SupMulti(sub_paths, sub_y, sup_tf_colo), batch_size=PROBE_BATCH, shuffle=True, num_workers=2, pin_memory=True)
        dl_va = DataLoader(SupMulti(va_paths, va_y, sup_tf_colo), batch_size=PROBE_BATCH, shuffle=False, num_workers=2, pin_memory=True)
        dl_te = DataLoader(SupMulti(test_paths, test_y, sup_tf_colo), batch_size=PROBE_BATCH, shuffle=False, num_workers=2, pin_memory=True)

        probe = train_probe_mc(encoder, dl_tr, dl_va, n_classes=len(class_names), seed=seed)
        metrics = eval_mc(probe, encoder, dl_te, n_classes=len(class_names))

        row = {
            "exp":"transfer_colorectal",
            "backbone":"resnet18",
            "method":"simclr",
            "pretrain":"pcam",
            "target":"colorectal",
            "split_id":SPLIT_ID,
            "seed":seed,
            "label_frac":frac,
            **metrics
        }
        append_row_csv(RAW_PATH, row)
        print("seed", seed, "frac", frac, "metrics", metrics)

summarize_mean_std(
    RAW_PATH,
    group_cols=["exp","backbone","method","pretrain","target","label_frac"],
    metric_cols=["acc","auc","f1"],
    out_csv_path=f"{OUT_DIR}/results_summary_resnet_transfer_colorectal.csv",
    out_tex_path=f"{OUT_DIR}/results_summary_resnet_transfer_colorectal.tex"
)

print("Saved:", RAW_PATH)
